# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [2]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/selfconsistency_results.jsonl"
MAX_TOKENS  = 8192               # capped from 32768 for milestone time constraint
SEED        = 42                 # fixed for reproducibility
EVAL_N_MCQ  = 10                 # stratified eval subset
EVAL_N_FREE = 10
K           = 3                  # number of samples per question for self-consistency
SUBSET_PATH = "results/eval_subset.json"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.80,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=16,
    max_num_batched_tokens=8192,
    enforce_eager=True,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-03 18:35:48 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.8, 'max_num_batched_tokens': 8192, 'max_num_seqs': 16, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-03 18:35:49 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 05-03 18:35:49 [model.py:1678] Using max model len 16384
INFO 05-03 18:35:49 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-03 18:35:50 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-03 18:35:50 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-03 18:35:50 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compila

(EngineCore pid=43925) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:09<00:19,  9.78s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:26<00:13, 13.61s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:26<00:00,  7.64s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:26<00:00,  8.87s/it]
(EngineCore pid=43925) 


(EngineCore pid=43925) INFO 05-03 18:36:34 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 29.237250 seconds
(EngineCore pid=43925) INFO 05-03 18:36:41 [gpu_worker.py:436] Available KV cache memory: 2.96 GiB
(EngineCore pid=43925) INFO 05-03 18:36:41 [kv_cache_utils.py:1319] GPU KV cache size: 21,536 tokens
(EngineCore pid=43925) INFO 05-03 18:36:41 [kv_cache_utils.py:1324] Maximum concurrency for 16,384 tokens per request: 1.31x
(EngineCore pid=43925) INFO 05-03 18:36:41 [core.py:283] init engine (profile, create kv cache, warmup model) took 6.89 seconds
(EngineCore pid=43925) INFO 05-03 18:36:43 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=43925) WARNING 05-03 18:36:43 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-03 18:36:43 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
(EngineCore pid=43

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [7]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)
#
# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)
#
# responses = [out.outputs[0].text.strip() for out in outputs]
#
# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


In [8]:
import hashlib
from pathlib import Path

# ── Build (or load) the stratified eval subset ─────────────────────────────
if Path(SUBSET_PATH).exists():
    with open(SUBSET_PATH) as f:
        subset_ids = json.load(f)
    print(f"Loaded existing subset of {len(subset_ids)} IDs from {SUBSET_PATH}")
else:
    import random
    rng = random.Random(SEED)

    def stratify_by_length(items, n):
        items = sorted(items, key=lambda d: len(d["question"]))
        b = len(items) // 3
        buckets = [items[:b], items[b:2*b], items[2*b:]]
        base, rem = divmod(n, 3)
        chosen = []
        for i, bucket in enumerate(buckets):
            k = base + (1 if i < rem else 0)
            chosen.extend(rng.sample(bucket, k))
        return chosen

    mcq_pool  = [d for d in data if d.get("options")]
    free_pool = [d for d in data if not d.get("options")]
    chosen = stratify_by_length(mcq_pool, EVAL_N_MCQ) + stratify_by_length(free_pool, EVAL_N_FREE)
    subset_ids = [d["id"] for d in chosen]

    Path(SUBSET_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(SUBSET_PATH, "w") as f:
        json.dump(subset_ids, f, indent=2)
    print(f"Saved new subset of {len(subset_ids)} IDs to {SUBSET_PATH}")

id_set = set(subset_ids)
subset = [d for d in data if d["id"] in id_set]
n_mcq  = sum(1 for d in subset if d.get("options"))
print(f"Eval subset: {len(subset)} questions ({n_mcq} MCQ, {len(subset)-n_mcq} free-form)")

# ── Cache: K samples per question (cache key includes K so changing K invalidates) ─
prompt_hash = hashlib.md5(
    (SYSTEM_PROMPT_MATH + "||" + SYSTEM_PROMPT_MCQ).encode()
).hexdigest()[:8]
CACHE_PATH = f"results/cache/{prompt_hash}_seed{SEED}_K{K}.jsonl"
Path(CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)

cache = {}
if Path(CACHE_PATH).exists():
    with open(CACHE_PATH) as f:
        for line in f:
            e = json.loads(line)
            # Only count as a hit if we have at least K samples
            if len(e.get("responses", [])) >= K:
                cache[e["id"]] = e["responses"][:K]
print(f"Cache hits: {len(cache)} for prompt hash {prompt_hash} K={K}")

to_generate = [item for item in subset if item["id"] not in cache]
print(f"Need to generate: {len(to_generate)} questions x {K} samples = {len(to_generate)*K} responses")
print(f"(MAX_TOKENS={MAX_TOKENS}, this will take roughly {K}x baseline time)")

if to_generate:
    prompts = []
    for item in to_generate:
        system, user = build_prompt(item["question"], item.get("options"))
        prompts.append(tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        ))

    # n=K → vLLM generates K diverse samples per prompt in one batched call.
    # Each sample within a group is internally seeded differently, so they're
    # genuinely distinct even with the same global seed.
    seeded_params = SamplingParams(
        max_tokens=MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        n=K,
        seed=SEED,
    )
    outputs = llm.generate(prompts, sampling_params=seeded_params)

    with open(CACHE_PATH, "a") as f:
        for item, out in zip(to_generate, outputs):
            resps = [o.text.strip() for o in out.outputs]   # K strings
            cache[item["id"]] = resps
            f.write(json.dumps({"id": item["id"], "responses": resps}) + "\n")
    print(f"Generated and cached {len(to_generate)} questions with {K} samples each")

# Now responses is a list-of-lists: responses[i] is K samples for subset[i]
responses = [cache[item["id"]] for item in subset]
print(f"Ready: {len(responses)} questions x {K} samples each")

# Preview
for i in range(min(2, len(responses))):
    print(f"\n── Question id={subset[i]['id']} (showing {min(K,2)}/{K} samples) ──")
    for k in range(min(K, 2)):
        snippet = responses[i][k][:200]
        print(f"  [sample {k}]: {snippet}{'...' if len(responses[i][k]) > 200 else ''}")


Loaded existing subset of 20 IDs from results/eval_subset.json
Eval subset: 20 questions (10 MCQ, 10 free-form)
Cache hits: 0 for prompt hash a761d123 K=3
Need to generate: 20 questions x 3 samples = 60 responses
(MAX_TOKENS=8192, this will take roughly 3x baseline time)


Rendering prompts:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/60 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=43925) INFO 05-03 19:58:23 [core.py:1210] Shutdown initiated (timeout=0)
(EngineCore pid=43925) INFO 05-03 19:58:23 [core.py:1215] Aborting 44 requests
(EngineCore pid=43925) INFO 05-03 19:58:23 [core.py:1233] Shutdown complete


KeyboardInterrupt: 

[rank0]:[W503 19:58:25.416981870 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


ERROR 05-03 19:58:28 [core_client.py:667] Engine core proc EngineCore died unexpectedly, shutting down client.


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
from collections import Counter

# Try utils.extract_boxed_answer for free-form answer extraction; fall back if signature differs
try:
    from utils import extract_boxed_answer
    def _norm_free(text):
        try:
            return (extract_boxed_answer(text) or "").strip()
        except Exception:
            m = re.search(r"\\boxed\{([^}]*)\}", text)
            return m.group(1).strip() if m else ""
except ImportError:
    def _norm_free(text):
        m = re.search(r"\\boxed\{([^}]*)\}", text)
        return m.group(1).strip() if m else ""

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def extract_answer_key(text, is_mcq):
    """Return a comparable string key for voting."""
    return extract_letter(text) if is_mcq else _norm_free(text)

def majority_vote(samples, is_mcq):
    """Return (winning_key, full_response_that_produced_it)."""
    keys = [extract_answer_key(s, is_mcq) for s in samples]
    counts = Counter(keys)
    winner_key, _ = counts.most_common(1)[0]
    for s, k in zip(samples, keys):
        if k == winner_key:
            return winner_key, s, keys
    return winner_key, samples[0], keys

# Load Judger
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

def judge_one(response, gold, is_mcq):
    if is_mcq:
        return extract_letter(response) == str(gold).strip().upper()
    gold_list = gold if isinstance(gold, list) else [gold]
    try:
        return judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
    except Exception:
        return False

results = []
for item, samples in tqdm(zip(subset, responses), total=len(subset), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    # Majority vote
    voted_key, voted_response, all_keys = majority_vote(samples, is_mcq)
    voted_correct = judge_one(voted_response, gold, is_mcq)

    # Per-sample correctness (for pass@1 and pass@K diagnostics)
    individual_correct = [judge_one(s, gold, is_mcq) for s in samples]

    # Agreement = fraction of K samples that produced the winning answer
    agreement = all_keys.count(voted_key) / len(all_keys)

    results.append({
        "id":        item.get("id"),
        "is_mcq":    is_mcq,
        "gold":      gold,
        "samples":   samples,
        "vote_keys": all_keys,
        "voted_key": voted_key,
        "voted_response": voted_response,
        "correct":   voted_correct,             # PRIMARY metric (maj@K)
        "individual_correct": individual_correct,
        "agreement": agreement,
    })

print(f"Scoring complete. {len(results)} results.")


## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

def pass_at_1(subset):
    """Average correctness across all K samples (i.e. expected single-sample accuracy)."""
    if not subset: return 0.0
    total = sum(sum(r["individual_correct"]) for r in subset)
    n = sum(len(r["individual_correct"]) for r in subset)
    return total / n * 100

def pass_at_k(subset):
    """Oracle: was at least one of the K samples correct? Upper bound, not usable at submission."""
    if not subset: return 0.0
    return sum(any(r["individual_correct"]) for r in subset) / len(subset) * 100

print("=" * 60)
print(f"SELF-CONSISTENCY RESULTS  (K={K} samples per question)")
print("=" * 60)
print(f"{'Metric':<20} {'MCQ':>10} {'Free':>10} {'Overall':>10}")
print("-" * 60)
print(f"{'pass@1 (avg)':<20} {pass_at_1(mcq_res):>9.2f}% {pass_at_1(free_res):>9.2f}% {pass_at_1(results):>9.2f}%")
print(f"{f'maj@{K} (vote)':<20} {acc(mcq_res):>9.2f}% {acc(free_res):>9.2f}% {acc(results):>9.2f}%")
print(f"{f'pass@{K} (oracle)':<20} {pass_at_k(mcq_res):>9.2f}% {pass_at_k(free_res):>9.2f}% {pass_at_k(results):>9.2f}%")
print("=" * 60)
print("\nReadings:")
print("  pass@1   = average accuracy of a single sample (comparable to baseline)")
print(f"  maj@{K}    = accuracy when majority-voting over K samples (the headline number)")
print(f"  pass@{K}   = accuracy if an oracle could pick the best of K (upper bound)")

# Per-length-bucket maj@K
sorted_q = sorted(subset, key=lambda d: len(d["question"]))
bsize = len(sorted_q) // 3
length_buckets = {
    "short ": [d["id"] for d in sorted_q[:bsize]],
    "medium": [d["id"] for d in sorted_q[bsize:2*bsize]],
    "long  ": [d["id"] for d in sorted_q[2*bsize:]],
}
print(f"\nmaj@{K} by question length:")
for label, ids in length_buckets.items():
    s = [r for r in results if r["id"] in set(ids)]
    if s:
        print(f"  {label}: {sum(r['correct'] for r in s):2d} / {len(s):2d}  ({acc(s):.2f}%)  agreement={sum(r['agreement'] for r in s)/len(s):.2f}")


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {
                "id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                "response": r["voted_response"],
                "correct": r["correct"],
                "voted_key": r["voted_key"],
                "vote_keys": r["vote_keys"],
                "individual_correct": r["individual_correct"],
                "agreement": r["agreement"],
                "K": K,
            }
        else:
            # Submission format only needs the chosen response
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["voted_response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!